# Copyright 2025 Google LLC

Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at

 https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.

#About this Colab

This script is designed to update the Default Language and Default Audio Language of YouTube videos in bulk. It retrieves a list of videos from a Google Spreadsheet and automatically updates the language settings for each video sequentially via the YouTube Data API.


###**Preparation Steps**

**1. Create a Google Gloud Project**
 *   Create a Google Cloud project if you don't already have one.  Follow [instructions](https://developers.google.com/workspace/guides/create-project) on how to set it up.

**2. Enable APIs**

 * Enable the [YouTube Data API v3](https://console.cloud.google.com/marketplace/product/google/youtube.googleapis.com)
 * Enable the [Google Sheets API](https://console.cloud.google.com/marketplace/product/google/sheets.googleapis.com)


**3 Create a Service account (CMS Level)**

* Create a service account in the 'Service Accounts' section of 'IAM & Admin' and a service account key for it using the [following tutorial](https://cloud.google.com/iam/docs/keys-create-delete).
* Get the email of the created Service Account. It ends with iam.gserviceaccount.com.
* Then generate a JSON key file for this service account using the [following tutorial](https://developers.google.com/workspace/guides/create-credentials#create_credentials_for_a_service_account).  It automatically downloads the key. (Section: Create Credentials for service account)
* Add your service account email as 'Video manager' to your CMS using the [following tutorial](https://support.google.com/youtube/answer/4524878).  It is automaticall enabled without the confirmation of the invitation.
* Add this service account to the Google Spreadsheet with the target video list as an editor.

**4. Prepare a Target Video List**

Paste the list of target videos for language change under the external_video_id column in the spreadsheet.

# Input Variables

In [ ]:
# Spreadsheet URL which the metadata will be stored
SPREADSHEET_URL = ''  # @param {type:"string"}

# Spreadsheet URL which the metadata will be stored
SHEET_NAME = 'Sheet1'  # @param {type:"string"}

# TARGET Audio Language to be changed. i.e. ko
TARGET_AUDIO_LANGUAGE = ''  # @param {type:"string"}

#Authentication and Library Setup

In [ ]:
from google.oauth2 import service_account
from google.colab import files
import json
from googleapiclient.discovery import build

# scope, name, version for YouTube Data API and Google Sheet API
YOUTUBE_DATA_API_SCOPES = ["https://www.googleapis.com/auth/youtube.force-ssl"]
GOOGLE_SHEET_API_SCOPE = ["https://www.googleapis.com/auth/spreadsheets"]
API_SERVICE_NAME = "youtube"
API_VERSION = "v3"

# Load json file
def service_account_loads():
  service_account_upload = files.upload()
  service_account_json = json.loads(
    next(iter(service_account_upload.values()))
  )
  return service_account_json

service_account_json = service_account_loads()

# Build YouTube Data API Service
youtube_data_credentials = service_account.Credentials.from_service_account_info(
      service_account_json, scopes=YOUTUBE_DATA_API_SCOPES)
youtube_service = build(
        API_SERVICE_NAME,
        API_VERSION,
        credentials=youtube_data_credentials,
    )

# Get Credential For Google Spreadsheet
spreadsheet_credentials = service_account.Credentials.from_service_account_info(
      service_account_json, scopes=GOOGLE_SHEET_API_SCOPE)

# Main Process

1. Open the target spreadsheet
2. Retrieve the individual video ids
3. Update the defaultLanguage and defaultAudioLanguage of the video
4. Display the result

In [ ]:
import gspread
import pandas as pd
import googleapiclient.errors

# open a spreadsheet with the target video list
def open_spreadsheet():
  print('Opening the spreadsheet.')
  gc = gspread.authorize(spreadsheet_credentials)

  try:
    worksheet = gc.open_by_url(SPREADSHEET_URL).worksheet(SHEET_NAME)
    all_data = worksheet.get_all_records()
    if not all_data:
      print('No data found in the spreadsheet.')
      return None
    df = pd.DataFrame(all_data)
    return df
  except Exception as e:
    print(f"An unexpected error occurred while reading the file: {e}")
    return None


# Send a request to update a video language
def request_video_language_update(video_resource_body):
  try:
    update_request = youtube_service.videos().update(
        part="snippet",
        body=video_resource_body
    )
    update_request.execute()

    print(f"SUCCESS: Video ID {video_id} updated (Default='{TARGET_AUDIO_LANGUAGE}', Audio='{TARGET_AUDIO_LANGUAGE}'.")
    return True

  except googleapiclient.errors.HttpError as e:
    print(f"FAILED: Error updating video (Code: {e.resp.status}).")
    if e.resp.status == 403:
          print("Reason: Permission denied. Check video ownership or API scope.")
    return None
  except Exception as e:
    print(f"An unexpected error occurred: {e}")
    return None


# Update defaultLanguage and defaultAudioLanguage of a video
def update_video_language(video_id):
  # Get current video snippet (required to include mandatory fields in update)
  try:
    get_request = youtube_service.videos().list(part="snippet", id=video_id)
    get_response = get_request.execute()

    if not get_response.get("items"):
      print(f"Video ID '{video_id}' not found.")
      return None

    current_snippet = get_response["items"][0]["snippet"]
    if current_snippet['defaultLanguage'] == TARGET_AUDIO_LANGUAGE and current_snippet['defaultAudioLanguage'] == TARGET_AUDIO_LANGUAGE:
      print(f"Video ID '{video_id}' is already in Korean.")
      return None

    # Construct and execute the update request
    video_resource_body = {
        "id": video_id,
        "snippet": {
            # MANDATORY fields to include:
            "title": current_snippet["title"],
            "description": current_snippet["description"],
            "categoryId": current_snippet["categoryId"],
            # Language fields to update:
            "defaultLanguage": TARGET_AUDIO_LANGUAGE,
            "defaultAudioLanguage": TARGET_AUDIO_LANGUAGE
        }
    }

    result =  request_video_language_update(video_resource_body)
    return result

  except googleapiclient.errors.HttpError as e:
    print(f"FAILED: Error Getting video data (Code: {e.resp.status}).")
    if e.resp.status == 403:
          print("Reason: Permission denied. Check video ownership or API scope.")
    return None
  except Exception as e:
    print(f"An unexpected error occurred with Video ID '{video_id}' : {e}")
    return None


if __name__ == '__main__':

  count = 0

  # Open shpreadsheet
  df = open_spreadsheet()

  # Load video list to memory
  videos = df['external_video_id']
  video_list = videos.tolist()
  print(f"Successfully loaded {len(video_list)} initial rows.")

  # Retrieve the individual video ids and update the language
  for video_id in video_list:
    result = update_video_language(video_id)
    if not result:
      continue
    else:
      count += 1

  # display the count of successfully updated videos
  print(f"Total {count} videos were successfully updated.")